In [16]:

import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

# 1. Cargar las dos bases
base = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_turnover.xlsx"
)

den = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/denuncias.xlsx"
)

In [17]:
# 2. Suma de delitos por ubigeo
delitos_sum = (
    den.groupby("ubigeo")["cantidad"]
       .sum()
       .reset_index()
       .rename(columns={"cantidad": "suma_delitos"})
)

# 3. Unir por ubigeo
final = base.merge(delitos_sum, on="ubigeo", how="left")

# 4. Guardar base final
final.to_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_final.xlsx",
    index=False
)

In [18]:

# quedarte solo con 2022 (donde existe turnover)
df_2022 = final[final["año"] == 2022].copy()

# Asegurar numéricos
num_cols = ["turnover_org", "suma_delitos", "orden_aparicion"]
for c in num_cols:
    df_2022[c] = pd.to_numeric(df_2022[c], errors="coerce")
    
    
# Mantener solo filas completas para el modelo
df_iv = df_2022[
    df_2022["turnover_org"].notna()
    & df_2022["suma_delitos"].notna()
    & df_2022["orden_aparicion"].notna()
    & df_2022["provincia"].notna()
].copy()

print("N (usadas en IV):", df_iv.shape[0])


N (usadas en IV): 5010


In [43]:
# ------------------------------------------------------------
# 2) OLS 
# ------------------------------------------------------------
ols_model = smf.ols(
    formula="suma_delitos ~ turnover_org",
    data=df_iv
).fit(cov_type="HC1")

print("\n================= OLS =================")
print(ols.summary())




================= OLS =================
                            OLS Regression Results                            
Dep. Variable:           suma_delitos   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     22.50
Date:                Sat, 07 Feb 2026   Prob (F-statistic):           2.16e-06
Time:                        02:13:36   Log-Likelihood:                -53771.
No. Observations:                5010   AIC:                         1.075e+05
Df Residuals:                    5008   BIC:                         1.076e+05
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Interce

In [42]:
# ------------------------------------------------------------
# 3) IV / 2SLS 
#   y = turnover_org
#   endog = suma_delitos
#   instrument = orden_aparicion
# ------------------------------------------------------------
iv = IV2SLS.from_formula(
    "suma_delitos ~ 1 + [turnover_org ~ orden_aparicion]",
    data=df_iv
).fit(cov_type="robust")

print("\n================= IV (2SLS) =================")
print(iv.summary)



================= IV (2SLS) =================
                          IV-2SLS Estimation Summary                          
Dep. Variable:           suma_delitos   R-squared:                     -10.263
Estimator:                    IV-2SLS   Adj. R-squared:                -10.265
No. Observations:                5010   F-statistic:                    9.2963
Date:                Sat, Feb 07 2026   P-value (F-stat)                0.0023
Time:                        02:13:25   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
Intercept     1.116e+04     2576.2     4.3328     0.0000      6112.7   1.621e+

In [32]:
# ------------------------------------------------------------
# 4) First stage 
# ------------------------------------------------------------
print("\n================= FIRST STAGE =================")
print(iv.first_stage)



================= FIRST STAGE =================
     First Stage Estimation Results    
                           turnover_org
---------------------------------------
R-squared                        0.0014
Partial R-squared                0.0014
Shea's R-squared                 0.0014
Partial F-statistic              11.047
P-value (Partial F-stat)         0.0009
Partial F-stat Distn            chi2(1)
==========================  ===========
Intercept                        0.0588
                               (11.405)
orden_aparicion                 -0.0033
                              (-3.3237)
---------------------------------------

T-stats reported in parentheses
T-stats use same covariance type as original model


In [41]:
# ------------------------------------------------------------
# 6) Diagnósticos 
# ------------------------------------------------------------
def safe_print(label, obj):
    print(f"\n--- {label} ---")
    print(obj)

def run_iv_diagnostics(iv_res):
    out = {}

    # Weak instruments / identification tests
    for attr in [
        "weak_instrument_test",     # KP/Cragg-Donald según versión/config
        "underidentification_test", # puede no existir según versión
        "anderson_rubin",           # robusto a instrumentos débiles
        "stock_wright",             # robusto a instrumentos débiles
    ]:
        try:
            v = getattr(iv_res, attr)
            v = v() if callable(v) else v
            out[attr] = v
        except Exception as e:
            out[attr] = f"NO DISPONIBLE / NO APLICA: {e}"

    # Endogeneity tests (justifica IV)
    for fn in ["durbin", "wu_hausman"]:
        try:
            v = getattr(iv_res, fn)
            v = v() if callable(v) else v
            out[fn] = v
        except Exception as e:
            out[fn] = f"NO DISPONIBLE / NO APLICA: {e}"

    # Overidentification tests (solo si sobreidentificado)
    for attr in ["sargan", "wooldridge_overid", "basmann"]:
        try:
            v = getattr(iv_res, attr)
            v = v() if callable(v) else v
            out[attr] = v
        except Exception as e:
            out[attr] = f"NO DISPONIBLE / NO APLICA: {e}"

    return out

diag = run_iv_diagnostics(iv)

print("\n================= DIAGNOSTICS (IV) =================")
for k, v in diag.items():
    safe_print(k, v)



================= DIAGNOSTICS (IV) =================

--- weak_instrument_test ---
NO DISPONIBLE / NO APLICA: 'IVResults' object has no attribute 'weak_instrument_test'

--- underidentification_test ---
NO DISPONIBLE / NO APLICA: 'IVResults' object has no attribute 'underidentification_test'

--- anderson_rubin ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Anderson-Rubin test of overidentification

--- stock_wright ---
NO DISPONIBLE / NO APLICA: 'IVResults' object has no attribute 'stock_wright'

--- durbin ---
Durbin test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 70.6073
P-value: 0.0000
Distributed: chi2(1)

--- wu_hausman ---
Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 71.5737
P-value: 0.0000
Distributed: F(1,5007)

--- sargan ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Sargan's test of overidentification

--- wooldridge_overid ---
Invalid

In [40]:
# ------------------------------------------------------------
# 7) Overidentificación / AR / etc.
# ------------------------------------------------------------
print("\n================= OVERID =================")
for attr in ["sargan", "wooldridge_overid", "basmann", "anderson_rubin"]:
    try:
        print(f"\n--- {attr} ---")
        print(getattr(iv, attr))
    except Exception as e:
        print(f"{attr}: no aplica/no disponible ({e})")


================= OVERID =================

--- sargan ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Sargan's test of overidentification

--- wooldridge_overid ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Wooldridge's score test of overidentification

--- basmann ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Basmann's test of overidentification

--- anderson_rubin ---
Invalid test statistic
Test requires more instruments than endogenous variables.
Anderson-Rubin test of overidentification


In [38]:
# ------------------------------------------------------------
# 8) Exportar a TXT/HTML para anexar al PDF
# ------------------------------------------------------------
with open("01_ols_Ydelitos.txt", "w", encoding="utf-8") as f:
    f.write(str(ols.summary()))

with open("02_iv_Ydelitos.txt", "w", encoding="utf-8") as f:
    f.write(str(iv.summary))

with open("03_first_stage_Ydelitos.txt", "w", encoding="utf-8") as f:
    f.write(str(iv.first_stage))

with open("04_reduced_form_Ydelitos.txt", "w", encoding="utf-8") as f:
    f.write(str(rf.summary()))

# HTML (si tu entorno lo permite)
try:
    with open("02_iv_Ydelitos.html", "w", encoding="utf-8") as f:
        f.write(iv.summary.as_html())
    with open("01_ols_Ydelitos.html", "w", encoding="utf-8") as f:
        f.write(ols.summary().as_html())
    with open("04_reduced_form_Ydelitos.html", "w", encoding="utf-8") as f:
        f.write(rf.summary().as_html())
except Exception as e:
    print("Export HTML no disponible:", e)

print("\nListo: 01_ols_Ydelitos.txt, 02_iv_Ydelitos.txt, 03_first_stage_Ydelitos.txt, 04_reduced_form_Ydelitos.txt (+ html si se pudo).")


Listo: 01_ols_Ydelitos.txt, 02_iv_Ydelitos.txt, 03_first_stage_Ydelitos.txt, 04_reduced_form_Ydelitos.txt (+ html si se pudo).
